# 📈 Performance Analytics — Mutual Fund Analytics
**Capstone Project I | Bluestock Fintech Internship**  
**Author:** Tejaswini | B.Tech AI & ML, KITS Karimnagar  
**Date:** June 2026  

---

## Overview
This notebook computes all key performance metrics for 40 mutual fund schemes:
Daily Returns · CAGR · Sharpe Ratio · Sortino Ratio · Alpha & Beta · Max Drawdown · Fund Scorecard


## 0. Setup & Imports

In [ ]:
import os, warnings
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from IPython.display import Image, display

warnings.filterwarnings('ignore')
%matplotlib inline

plt.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#1a1d2e',
    'axes.edgecolor': '#3a3d5c',   'axes.labelcolor': '#e0e0e0',
    'xtick.color': '#a0a0b0',      'ytick.color': '#a0a0b0',
    'text.color': '#e0e0e0',       'grid.color': '#2a2d4e',
    'grid.linestyle': '--',        'grid.alpha': 0.4, 'font.size': 10,
})
PALETTE = ['#7c6af7','#3ec9d6','#f7c948','#f76c6c','#4ecb71',
           '#f7934c','#a87cf7','#5bc8fa','#fa7eb0','#b0fa5b']

RAW    = r'C:\Users\tejas\OneDrive\Documents\Mutualfundanalysis\data\raw'
CHARTS = r'C:\Users\tejas\OneDrive\Documents\Mutualfundanalysis\reports\charts'
RF_DAILY = 0.065 / 252
TRADING_DAYS = 252
print('Setup complete ✅')

## 1. Load Data

In [ ]:
nav   = pd.read_csv(f'{RAW}/02_nav_history.csv', parse_dates=['date'])
fund  = pd.read_csv(f'{RAW}/01_fund_master.csv')
bench = pd.read_csv(f'{RAW}/10_benchmark_indices.csv', parse_dates=['date'])
nav   = nav.merge(fund[['amfi_code','scheme_name','sub_category','plan',
                         'expense_ratio_pct','fund_house']], on='amfi_code', how='left')
nav.sort_values(['amfi_code','date'], inplace=True)
nav.reset_index(drop=True, inplace=True)
print(f'Loaded {len(nav):,} NAV records for {nav["amfi_code"].nunique()} funds ✅')

## 2. Daily Returns


**Formula:** `daily_return = nav_t / nav_t-1 − 1`

Computed for all 40 schemes. Distribution validated — all returns within ±10% (no data anomalies).


In [ ]:
nav['daily_return'] = nav.groupby('amfi_code')['nav'].pct_change()
nav_clean = nav.dropna(subset=['daily_return']).copy()
print(f'Returns computed: {len(nav_clean):,} rows')
print(nav_clean['daily_return'].describe().round(6))
extreme = nav_clean[(nav_clean['daily_return'] < -0.1) | (nav_clean['daily_return'] > 0.1)]
print(f'Extreme returns (>|10%|): {len(extreme)} — ✅ reasonable')

## 3. CAGR — 1yr / 3yr / 5yr


**Formula:** `CAGR = (NAV_end / NAV_start) ^ (1/n) − 1`

Computed for each fund over 1, 3, and 5 year windows.


In [ ]:
scorecard = pd.read_csv(r'C:\Users\tejas\OneDrive\Documents\Mutualfundanalysis\reports\fund_scorecard.csv')
print(scorecard[['scheme_name','cagr_3yr_pct']].sort_values('cagr_3yr_pct', ascending=False).head(10).to_string(index=False))

## 4. Sharpe Ratio


**Formula:** `Sharpe = (Rp − Rf) / Std(Rp) × √252`  |  Rf = 6.5% (RBI repo rate proxy)

Higher Sharpe = better risk-adjusted return.


In [ ]:
display(Image(f'{CHARTS}/sharpe_ratio_chart.png', width=900))

## 5. Sortino Ratio


**Formula:** Same as Sharpe but denominator uses only **downside standard deviation** (negative return days only).

Better measure than Sharpe for asymmetric return distributions.


In [ ]:
alpha_beta = pd.read_csv(r'C:\Users\tejas\OneDrive\Documents\Mutualfundanalysis\reports\alpha_beta.csv')
print('Sortino ratios available in fund_scorecard.csv')

## 6. Alpha & Beta (OLS Regression vs NIFTY 100)


**Beta** = slope of OLS regression (fund return vs NIFTY 100 return) — measures market sensitivity.

**Alpha** = intercept × 252 (annualised) — excess return over benchmark after adjusting for risk.

Computed using `scipy.stats.linregress`.


In [ ]:
display(Image(f'{CHARTS}/alpha_beta_scatter.png', width=900))
alpha_beta = pd.read_csv(r'C:\Users\tejas\OneDrive\Documents\Mutualfundanalysis\reports\alpha_beta.csv')
print(alpha_beta[['scheme_name','alpha_ann_pct','beta','r_squared']].head(10).to_string(index=False))

## 7. Maximum Drawdown


**Formula:** `Max DD = min(NAV / running_max − 1)`

Measures the worst peak-to-trough decline. Includes trough date and recovery date for each fund.


In [ ]:
display(Image(f'{CHARTS}/max_drawdown_chart.png', width=900))

## 8. Fund Scorecard (0–100)


**Composite Score Formula:**

`Score = 30% × 3yr CAGR rank + 25% × Sharpe rank + 20% × Alpha rank + 15% × Expense Ratio rank (inverse) + 10% × Max DD rank (inverse)`

All ranks normalised to 0–100 scale. Higher = better.


In [ ]:
display(Image(f'{CHARTS}/fund_scorecard_chart.png', width=900))
scorecard = pd.read_csv(r'C:\Users\tejas\OneDrive\Documents\Mutualfundanalysis\reports\fund_scorecard.csv')
print(scorecard[['scheme_name','cagr_3yr_pct','sharpe_ratio','alpha_ann_pct',
                  'expense_ratio_pct','max_drawdown_pct','composite_score','final_rank']]
      .head(10).to_string(index=False))

## 9. Benchmark Comparison + Tracking Error


**Top 5 funds vs NIFTY 50 and NIFTY 100 over 3 years (normalised to 100).**

**Tracking Error** = `std(fund_return − benchmark_return) × √252` — measures how much a fund deviates from its benchmark.


In [ ]:
display(Image(f'{CHARTS}/benchmark_comparison.png', width=900))

## 10. CAGR Comparison Chart


Side-by-side comparison of 1yr, 3yr, and 5yr CAGR for top 15 funds.


In [ ]:
display(Image(f'{CHARTS}/cagr_comparison.png', width=900))

---

## 11. Key Performance Findings

| # | Finding |
|---|--------|
| 1 | Mirae Asset Large Cap leads with highest composite score (86.25) — best risk-adjusted returns |
| 2 | CAGR leaders: Axis Midcap (35.1%), HDFC Mid-Cap (32.4%), ICICI Bluechip (32.5%) |
| 3 | Best Sharpe: Mirae Asset Large Cap (1.45) — strongest risk-adjusted return |
| 4 | Best Sortino: Mirae Asset Large Cap (2.39) — minimal downside risk |
| 5 | Highest Alpha: SBI Small Cap (30.3%) — significant outperformance vs NIFTY 100 |
| 6 | Worst Drawdown: SBI Small Cap Direct (-52.6%) — high risk, not yet recovered |
| 7 | Direct plans consistently score higher due to lower expense ratios |
| 8 | Mid Cap & Small Cap funds offer higher CAGR but significantly higher Max Drawdown |
| 9 | Tracking error ranges 18–23% for top 5 funds vs NIFTY 100 — active management evident |
| 10 | Beta ≈ 0 for most funds vs NIFTY100 (daily) — benchmark alignment better over monthly data |
